# 🐄 Cattle Disease Diagnostic Application: ML Track Final MVP
**Project**: Hybrid Online-Offline Explainable Mobile Application for Early Detection of Priority Cattle Diseases in South Sudan.

### Objectives Implemented:
1. **Multimodal Fusion**: Combining image features with structured clinical symptoms.
2. **Data Engineering**: Balancing datasets for ECF and CBPP using clinical sign profiling.
3. **Model Architecture**: Optimized MobileNetV2 with a functional fusion layer.
4. **Edge Deployment**: Conversion to TensorFlow Lite with 8-bit quantization.



In [ ]:
import kagglehub
import os
import pandas as pd
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras import layers, models, Input

# Download primary vision dataset
path = kagglehub.dataset_download("devang03mgr/cattle-diseases-datasets")
dataset_path = os.path.join(path, 'Cows datasets')
print(f"Vision Dataset Path: {dataset_path}")



## 1. Data Engineering: Multimodal Dataset Construction
We map existing vision data (LSD, FMD, Healthy) to clinical profiles and engineer the missing classes (ECF, CBPP) using the specific signs identified in the research (e.g., Parotid Lymph Node swelling for ECF).



In [ ]:
# Define clinical sign profiles [fever, nodules, mouth_sores, nasal_discharge, cough, swollen_lymph]
disease_profiles = {
    'lumpy': [1, 1, 0, 1, 0, 0],           # LSD
    'foot-and-mouth': [1, 0, 1, 0, 0, 0],  # FMD
    'healthy': [0, 0, 0, 0, 0, 0],         # Healthy
    'CBPP': [1, 0, 0, 1, 1, 0],            # Contagious Bovine Pleuropneumonia
    'ECF': [1, 0, 0, 0, 0, 1]              # East Coast Fever
}

data_records = []

# Process Vision Data
for cls in ['lumpy', 'foot-and-mouth', 'healthy']:
    cls_path = os.path.join(dataset_path, cls)
    images = os.listdir(cls_path)
    for img in images[:400]: # Take 400 samples for robust training
        data_records.append({
            'image_path': os.path.join(cls, img),
            'fever': disease_profiles[cls][0],
            'nodules': disease_profiles[cls][1],
            'mouth_sores': disease_profiles[cls][2],
            'nasal_discharge': disease_profiles[cls][3],
            'cough': disease_profiles[cls][4],
            'swollen_lymph': disease_profiles[cls][5],
            'disease': cls.upper() if cls != 'lumpy' else 'LSD'
        })

# Synthesize missing ECF and CBPP using Healthy images as proxies (Multimodal logic)
healthy_imgs = os.listdir(os.path.join(dataset_path, 'healthy'))[400:800]
for i, img in enumerate(healthy_imgs):
    d_type = 'CBPP' if i < 200 else 'ECF'
    data_records.append({
        'image_path': os.path.join('healthy', img),
        'fever': disease_profiles[d_type][0],
        'nodules': disease_profiles[d_type][1],
        'mouth_sores': disease_profiles[d_type][2],
        'nasal_discharge': disease_profiles[d_type][3],
        'cough': disease_profiles[d_type][4],
        'swollen_lymph': disease_profiles[d_type][5],
        'disease': d_type
    })

df = pd.DataFrame(data_records)
df.to_csv('cattle_multimodal_data.csv', index=False)
print("Multimodal dataset engineered and saved.")



## 2. Data Visualization
Visualizing the distribution of the 5 priority classes to ensure class balance for training.



In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='disease', data=df, palette='Reds_d')
plt.title('Distribution of Priority Cattle Diseases')
plt.ylabel('Count of Samples')
plt.show()



In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['disease'])
num_classes = len(le.classes_)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

def preprocess_image(path_suffix):
    full_path = os.path.join(dataset_path, path_suffix)
    img = tf.io.read_file(full_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    return img / 255.0

def create_generator(dataframe, batch_size=32):
    def gen():
        for _, row in dataframe.iterrows():
            img = preprocess_image(row['image_path'])
            symptoms = row[['fever', 'nodules', 'mouth_sores', 'nasal_discharge', 'cough', 'swollen_lymph']].values.astype('float32')
            label = row['label']
            yield (img, symptoms), label
    
    return tf.data.Dataset.from_generator(
        gen,
        output_signature=((tf.TensorSpec(shape=(224, 224, 3), dtype=tf.float32), 
                           tf.TensorSpec(shape=(6,), dtype=tf.float32)), 
                          tf.TensorSpec(shape=(), dtype=tf.int32))
    ).batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_ds = create_generator(train_df)
val_ds = create_generator(val_df)



## 3. Model Architecture: Multimodal Fusion Layer
We use a Functional API to branch the Vision (MobileNetV2) and Symptom (Dense MLP) inputs before late fusion.



In [ ]:
# Vision Branch
image_in = Input(shape=(224, 224, 3), name='image_input')
base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base.trainable = False
x = base(image_in)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)

# Symptom Branch
symptom_in = Input(shape=(6,), name='symptom_input')
y = layers.Dense(32, activation='relu')(symptom_in)
y = layers.Dense(16, activation='relu')(y)

# Fusion
merged = layers.concatenate([x, y])
merged = layers.Dense(64, activation='relu')(merged)
merged = layers.Dropout(0.3)(merged)
output = layers.Dense(num_classes, activation='softmax')(merged)

model = models.Model(inputs=[image_in, symptom_in], outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()



In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=10)



## 4. Evaluation & Metrics
Generating the classification report and confusion matrix to validate diagnostic accuracy (Target >= 85%).



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for (img, sym), lbl in val_ds:
    p = model.predict((img, sym), verbose=0)
    y_true.extend(lbl.numpy())
    y_pred.extend(np.argmax(p, axis=1))

print(classification_report(y_true, y_pred, target_names=le.classes_))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix: Hybrid Diagnostic Model')
plt.show()



## 5. Deployment: Edge AI Optimization
Converting to TFLite with quantization to ensure the model size is < 50MB and runs with < 5s latency offline.



In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open('cattle_health_mvp.tflite', 'wb') as f:
    f.write(tflite_model)

print("TFLite Model Ready for Flutter Deployment.")

